# Amazon Fine Food Reviews: End-to-End Sentiment Analysis

## 1. Project Introduction

Customer reviews contain valuable information about product quality, customer satisfaction, and purchasing experiences. However, manually analysing hundreds of thousands of reviews is time-consuming and difficult to scale.

This project develops a Natural Language Processing (NLP) and machine learning pipeline to automatically classify Amazon Fine Food reviews into positive and negative sentiment categories.

The project explores multiple approaches to text representation and sentiment classification, beginning with traditional machine learning methods and progressing towards pretrained language representations and transformer fine-tuning.

### Project Objectives

1. Explore and understand the Amazon Fine Food Reviews dataset.
2. Convert numerical product ratings into binary sentiment labels.
3. Establish a majority-class baseline.
4. Develop TF-IDF-based sentiment classifiers using unigram and bigram features.
5. Investigate the effect of classification thresholds.
6. Experiment with pretrained GloVe word embeddings.
7. Explore Sentence Transformer embeddings.
8. Explore pretrained BERT representations through sentence similarity and sample review embeddings.
9. Fine-tune BERT for binary sentiment classification.
10. Analyse classification errors and prediction confidence.
11. Compare the approaches and select a suitable model for deployment.

**Scope clarification:** The pretrained BERT representation experiment demonstrates contextual embeddings and semantic similarity. These embeddings are not used to train a separate traditional machine learning classifier in this project. The BERT classification experiment is performed separately by fine-tuning `bert-base-uncased` for binary sentiment classification.

### Problem Formulation

This is a supervised binary text classification problem.

- **Input:** A customer review written in natural language.
- **Output:** A predicted sentiment label.
- **Class 0:** Negative sentiment.
- **Class 1:** Positive sentiment.

The final objective is not simply to maximise accuracy, but to develop a model that can reliably identify sentiment while considering the imbalance between positive and negative reviews.


## 2. Environment Setup

The project uses Python and several machine learning and NLP libraries.

The initial environment is configured to support:

- Data manipulation and numerical computation.
- Text feature extraction using TF-IDF.
- Logistic Regression classification.
- Model evaluation using classification metrics.
- Loading pretrained word embeddings through Gensim.

Gensim is installed because the project later explores GloVe embeddings as an alternative to sparse TF-IDF features.

The project also uses transformer-based libraries in later experiments for Sentence Transformers and BERT.

In [ ]:
!pip install gensim

## 3. Importing Libraries

The required libraries are imported before beginning the data analysis and modelling stages.

### Data Processing
- Pandas is used for loading and manipulating tabular data.
- NumPy supports numerical operations and array transformations.

### Machine Learning
- Scikit-learn provides Logistic Regression, train-test splitting, TF-IDF vectorization, and evaluation metrics.

### NLP
- Gensim is used to load pretrained GloVe word vectors.
- Transformer libraries are introduced later for contextual text representations.

### Evaluation
Accuracy, precision, recall, F1-score, classification reports, and confusion matrices are used to evaluate model performance.

These metrics provide a more complete assessment than accuracy alone, particularly because the dataset contains substantially more positive reviews than negative reviews.

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix,precision_score,recall_score,f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api

## 4. Loading the Amazon Fine Food Reviews Dataset

The Amazon Fine Food Reviews dataset is loaded into a Pandas DataFrame.

The dataset contains customer reviews of food products along with associated information, including:

- Product and user identifiers.
- Helpfulness numerator and denominator.
- Numerical product rating (`Score`).
- Review timestamp.
- Review summary.
- Full review text.

The two most important columns for this project are:

- `Score`: Used to derive the sentiment target.
- `Text`: Used as the input feature for the NLP models.

The full review text is selected rather than relying only on the summary because it generally contains more detailed information about the customer's experience.

In [4]:
df = pd.read_csv(
    "/kaggle/input/datasets/reetdubey/reviews/Reviews.csv"
)

### Initial Dataset Inspection

Display the first few records to understand how the dataset is structured.

This inspection helps verify that the file has been loaded correctly and that the expected columns are available.

The `Text` column contains the actual customer reviews, while `Score` contains the numerical ratings that will later be converted into sentiment labels.



In [5]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [6]:
df.shape

(568454, 10)

### Dataset Dimensions

The dataset contains 568,454 records and 10 columns.

This is a sufficiently large collection of reviews to investigate different text representation techniques and compare traditional machine learning models with pretrained language models.

The dataset size also influences the modelling strategy: computationally expensive transformer experiments are performed on smaller subsets, while the main TF-IDF experiments use the full training and test splits.

In [7]:
df.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time
count,568454.000000,568454.000000,568454.00000,568454.000000,5.684540e+05
mean,284227.500000,1.743817,2.22881,4.183199,1.296257e+09
std,164098.679298,7.636513,8.28974,1.310436,4.804331e+07
min,1.000000,0.000000,0.00000,1.000000,9.393408e+08
25%,142114.250000,0.000000,0.00000,4.000000,1.271290e+09
50%,284227.500000,0.000000,1.00000,5.000000,1.311120e+09
75%,426340.750000,2.000000,2.00000,5.000000,1.332720e+09
max,568454.000000,866.000000,923.00000,5.000000,1.351210e+09


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Id                      568454 non-null  int64 
 1   ProductId               568454 non-null  object
 2   UserId                  568454 non-null  object
 3   ProfileName             568428 non-null  object
 4   HelpfulnessNumerator    568454 non-null  int64 
 5   HelpfulnessDenominator  568454 non-null  int64 
 6   Score                   568454 non-null  int64 
 7   Time                    568454 non-null  int64 
 8   Summary                 568427 non-null  object
 9   Text                    568454 non-null  object
dtypes: int64(5), object(5)
memory usage: 43.4+ MB


In [9]:
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

### Missing-Value Analysis

Check each column for missing values before proceeding with modelling.

The dataset contains a small number of missing values in `ProfileName` and `Summary`, while the `Text` and `Score` columns used in the main classification task are complete.

Since the selected input feature and target column contain no missing values, there is no need to remove reviews or impute values for the main sentiment classification pipeline.

The unused metadata columns are retained in the original DataFrame but are not required as model inputs.

In [10]:
df["Score"].value_counts().sort_index()

Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64

## 5. Rating Distribution Analysis

Examine the frequency of each original product rating.

The Amazon Fine Food Reviews dataset uses a five-point rating scale:

- 1: Lowest rating.
- 2: Low rating.
- 3: Moderate rating.
- 4: High rating.
- 5: Highest rating.

Understanding the rating distribution is necessary before converting ratings into sentiment labels.

The dataset contains a particularly large number of five-star reviews, indicating that the resulting binary classification problem will be imbalanced.

## 6. Defining the Sentiment Classification Target

The original dataset contains five rating categories, whereas this project focuses on binary sentiment classification.

The following mapping is adopted:

| Original Rating | Sentiment | Interpretation |
|---|---:|---|
| 1 | 0 | Negative |
| 2 | 0 | Negative |
| 3 | 0 | Negative |
| 4 | 1 | Positive |
| 5 | 1 | Positive |

### Why This Mapping Was Chosen

Ratings of 1 and 2 clearly represent dissatisfaction, while ratings of 4 and 5 generally indicate satisfaction.

Three-star reviews are more ambiguous and may represent mixed experiences. In this project, they are grouped with negative reviews to retain all records while using a conservative definition of positive sentiment.

This is a modelling assumption rather than a perfect measurement of the emotional tone of every review. Some three-star reviews may contain positive language, and some four-star reviews may contain criticism.

The resulting labels may therefore contain a degree of label noise.

Nevertheless, this mapping creates a practical binary classification target without discarding the three-star reviews.

In [11]:
df["Sentiment"] = df["Score"].map({
    1: 0,
    2: 0,
    3: 0,
    4: 1,
    5: 1
})
df["Sentiment"].value_counts(normalize=True)

Sentiment
1    0.780674
0    0.219326
Name: proportion, dtype: float64

### Interpreting the Sentiment Distribution

The resulting class distribution is:

| Sentiment | Count | Proportion |
|---|---:|---:|
| Negative (0) | 124,677 | 21.93% |
| Positive (1) | 443,777 | 78.07% |

The dataset is clearly imbalanced, with positive reviews representing approximately 78% of all observations.

### Why This Matters

A classifier could achieve approximately 78% accuracy by predicting every review as positive.

Such a model would fail to identify negative reviews entirely.

Therefore, accuracy alone is not sufficient to assess model quality. Particular attention must be given to negative-class recall, precision, and F1-score, as well as the confusion matrix.

In [12]:
pd.crosstab(df["Score"], df["Sentiment"])

Sentiment,0,1
Score,,
1,52268,0
2,29769,0
3,42640,0
4,0,80655
5,0,363122


### Validating the Target Transformation

A cross-tabulation is used to verify that each original rating has been mapped to the intended sentiment class.

This is an important validation step because an incorrect target mapping would affect every subsequent training and evaluation result.

The expected relationship is:

- Ratings 1, 2, and 3 appear only under sentiment 0.
- Ratings 4 and 5 appear only under sentiment 1.

The cross-tabulation confirms that the mapping has been applied consistently.

In [13]:
X=df['Text']
y=df['Sentiment']

In [14]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

# 7. Establishing a Majority-Class Baseline

Before training a machine learning model, establish a simple baseline using the most frequent sentiment class in the training data.

The baseline classifier predicts the majority class for every review.

Since positive reviews represent approximately 78% of the dataset, this baseline predicts every test review as positive.

### Why Is a Baseline Necessary?

A baseline establishes the minimum performance that a more sophisticated model should outperform.

It also demonstrates why accuracy can be misleading in an imbalanced classification problem.

A model that achieves approximately 78% accuracy may appear reasonable, but if it cannot identify negative reviews, it has limited practical value.

In [15]:
majority_class = y_train.mode()[0]
baseline_pred = [majority_class] * len(y_test)
print(accuracy_score(y_test,baseline_pred))
print(precision_score(y_test,baseline_pred))
print(recall_score(y_test,baseline_pred))
print(classification_report(y_test,baseline_pred))
print(confusion_matrix(y_test,baseline_pred))

0.7806774502819045
0.7806774502819045
1.0
              precision    recall  f1-score   support

           0       0.00      0.00      0.00     24935
           1       0.78      1.00      0.88     88756

    accuracy                           0.78    113691
   macro avg       0.39      0.50      0.44    113691
weighted avg       0.61      0.78      0.68    113691

[[    0 24935]
 [    0 88756]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Baseline Results and Interpretation

The majority-class baseline achieves approximately **78.07% accuracy**.

However, its confusion matrix shows that every review is predicted as positive:

- True negatives: 0
- False positives: 24,935
- False negatives: 0
- True positives: 88,756

The model fails to identify any negative reviews.

Its negative-class precision, recall, and F1-score are therefore zero.

### Key Finding

The baseline demonstrates that accuracy alone is not an adequate performance measure for this dataset.

A useful sentiment classifier must learn linguistic patterns that distinguish negative reviews from positive reviews instead of simply predicting the dominant class.

This motivates the use of TF-IDF feature extraction and supervised classification.

# 8. Unigram TF-IDF Feature Extraction

Machine learning algorithms require numerical input, so the raw review text must first be converted into numerical features.

Term Frequency–Inverse Document Frequency (TF-IDF) represents the importance of a word within a document relative to its occurrence across the entire document collection.

### Why TF-IDF?

- It gives greater importance to words that are informative within a review.
- It reduces the influence of words that appear frequently across many reviews.
- It creates sparse numerical representations suitable for traditional machine learning models.
- It provides a computationally efficient baseline for large text datasets.

### Unigram Configuration

The first experiment uses:

- Lowercase conversion.
- `ngram_range=(1,1)`.
- `max_features=20000`.

A unigram is a single word considered independently.

For example, the sentence:

"The food was very good"

is represented using individual tokens such as `food`, `very`, and `good`.

The vocabulary is limited to 20,000 features to control dimensionality and computational cost.

In [16]:
tfidf=TfidfVectorizer(lowercase=True,ngram_range=(1,1),max_features=20000)

In [17]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [18]:
len(tfidf.get_feature_names_out())

20000

## 9. Unigram TF-IDF with Logistic Regression

Logistic Regression is selected as the first supervised classification algorithm.

Although its name contains the word regression, Logistic Regression is widely used for binary classification.

It learns a relationship between the TF-IDF features and the sentiment labels and estimates the probability that a review belongs to the positive class.

### Why Logistic Regression?

- It is computationally efficient for sparse text features.
- It works well with high-dimensional TF-IDF representations.
- It provides class probabilities that can be used for threshold experiments.
- Its coefficients can be inspected to understand influential words.

The model is configured with `max_iter=500` to allow sufficient optimization iterations during training.

In [19]:
model=LogisticRegression(max_iter=500)
model.fit(X_train_tfidf,y_train)
y_pred = model.predict(X_test_tfidf)

In [20]:
print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.9015577310429146
0.919364605635935
0.9579183379151832
              precision    recall  f1-score   support

           0       0.82      0.70      0.76     24935
           1       0.92      0.96      0.94     88756

    accuracy                           0.90    113691
   macro avg       0.87      0.83      0.85    113691
weighted avg       0.90      0.90      0.90    113691

[[17478  7457]
 [ 3735 85021]]


### Unigram Model: Results and Interpretation

The unigram TF-IDF model achieves the following performance on the full held-out test set:

| Metric | Score |
|---|---:|
| Accuracy | 90.16% |
| Precision (positive class) | 91.94% |
| Recall (positive class) | 95.79% |
| F1-score (positive class) | 93.82% |

The confusion matrix is:

| | Predicted Negative | Predicted Positive |
|---|---:|---:|
| Actual Negative | 17,478 | 7,457 |
| Actual Positive | 3,735 | 85,021 |

### Interpretation

The model correctly identifies a large proportion of positive reviews, achieving 95.79% positive-class recall.

However, it identifies only 70% of the negative reviews, as shown by the negative-class recall of approximately 0.70.

A total of 7,457 negative reviews are incorrectly classified as positive.

This indicates that the model is more effective at recognising positive sentiment than negative sentiment.

### Key Finding

Unigram TF-IDF provides a strong improvement over the majority-class baseline.

However, individual words may not capture the meaning of short phrases or word combinations. For example, the word `good` may appear in both positive and negative contexts.

This motivates the next experiment using bigram features.

## 10. Error Analysis: Unigram Model

Aggregate the test predictions with their corresponding actual labels and review text.

The objective is to inspect misclassified reviews rather than relying exclusively on numerical metrics.

Two types of classification errors are examined:

- **False Positive:** A negative review incorrectly predicted as positive.
- **False Negative:** A positive review incorrectly predicted as negative.

Examining actual review text can reveal why the model makes mistakes, especially when sentiment depends on context, negation, mixed opinions, or product-specific experiences.

In [21]:
error_df = pd.DataFrame({
    "text": X_test,
    "actual": y_test,
    "predicted": y_pred
})
errors = error_df[error_df["actual"] != error_df["predicted"]]
errors.shape

(11192, 3)

In [22]:
false_positives = errors[errors["actual"] == 0]
false_negatives = errors[errors["actual"] == 1]
false_positives["text"].sample(10, random_state=42).tolist()

["This is the type of Goji Juice that purists tell you not to get but I found it to be very tasty and even if it was mixed with other juices,I felt I was getting some health benefits. It's also a lot less $$$$$ than pure Goji, by about $15-$17 PER BOTTLE!!!!!",
 "I always get a hankering for gingerbread and pumpkin bread when Autumn rolls around, but I'm never up for baking from scratch.  This year was the first time I tried this particular mix.  It calls for one egg, some butter, and some milk to be added to the mix, so quite easy to throw together.<br /><br />The result is a bit of a mixed bag.  The texture and consistency is very lovely and moist; I'd grade it an A or A+ almost.  Unfortunately the sweetness is underplayed to the detriment of the whole project.  I have so far sampled it plain; I wonder if a scoop of whipped cream might mitigate the problem.  Anyway, I might buy it again, or I might not.",
 'I just wanted to let everyone know about hair oil products Bc I use them at t

In [23]:
false_negatives["text"].sample(10, random_state=42).tolist()

['I live in an area in Minnesota where mosquitoes are numerous and extremely bothersome.  Since we have relatively short summers, I look forward to the summer season to enjoy the outdoors, but mosquitoes will literally ruin the summer season if not treated.  In addition, mosquitoes carry West Nile Disease, and a dear friend several houses away contracted this disease while working in his yard.  He was severely ill for months, and was lucky to survive.  To protect my family, I decided to take my yard back from the blood-sucking, disease-carrying demoms.  I first tried one of the mosquitoe magnet devices, <a href="http://www.amazon.com/gp/product/B000094YKM">Mosquito Magnet MM4000 Defender Mosquito Trap</a>,but despite its great hype (and price!), it did not eliminated the biting tormentors.  I then sprayed the yard with an application of Cutter Bug-Free Backyard.  The results were stunning!  For the first time in many summer seasons, the yard was virtually free of mosquitoes. I have bee

### Inspecting False Positives and False Negatives

The unigram model produces 11,192 incorrect predictions on the full test set.

These errors demonstrate that even a model with approximately 90% accuracy can misclassify a substantial number of customer reviews when evaluated on a large dataset.

The next step separates false positives and false negatives to identify the main sources of classification difficulty.

False positives are negative reviews that the model predicts as positive.

False negatives are positive reviews that the model predicts as negative.

These examples are particularly useful because customer reviews frequently contain mixed sentiment.

For example, a review may praise a product's taste while criticising its price, packaging, quality, or delivery.

A unigram model may identify positive words without adequately considering the surrounding context.

The sampled reviews provide qualitative evidence of these limitations and motivate the use of phrase-level features.

# 11. Bigram TF-IDF Feature Extraction

The next experiment extends the unigram representation by including bigrams.

A bigram is a sequence of two consecutive words.

For example:

- Unigram features: `not`, `good`
- Bigram feature: `not good`

### Why Introduce Bigrams?

Individual words can have different meanings depending on the words surrounding them.

Consider:

- "The product is good."
- "The product is not good."

A unigram representation contains the word `good` in both sentences.

Including bigrams allows the model to represent the phrase `not good`, providing additional contextual information.

### Configuration

The bigram vectorizer uses:

- Lowercase conversion.
- `ngram_range=(1,2)`.
- `max_features=50000`.

The feature limit is increased to 50,000 to accommodate the additional phrase-level features.

This increases the dimensionality of the feature space and therefore requires more memory and computation, but it allows the classifier to learn richer linguistic patterns.

In [24]:
tfidf_bigram = TfidfVectorizer(lowercase=True,ngram_range=(1, 2),max_features=50000)

In [25]:
X_train_tfidf_bigram = tfidf_bigram.fit_transform(X_train)
X_test_tfidf_bigram = tfidf_bigram.transform(X_test)

In [26]:
model1=LogisticRegression(max_iter=500)
model1.fit(X_train_tfidf_bigram,y_train)
y_pred_bigram = model1.predict(X_test_tfidf_bigram)

In [27]:
print(accuracy_score(y_test, y_pred_bigram))
print(precision_score(y_test, y_pred_bigram))
print(recall_score(y_test, y_pred_bigram))
print(classification_report(y_test, y_pred_bigram))
print(confusion_matrix(y_test, y_pred_bigram))

0.9288246211221645
0.9402480024450945
0.9705034025868674
              precision    recall  f1-score   support

           0       0.88      0.78      0.83     24935
           1       0.94      0.97      0.96     88756

    accuracy                           0.93    113691
   macro avg       0.91      0.88      0.89    113691
weighted avg       0.93      0.93      0.93    113691

[[19461  5474]
 [ 2618 86138]]


### Bigram Model: Results and Interpretation

The bigram TF-IDF model achieves the following results on the full held-out test set:

| Metric | Score |
|---|---:|
| Accuracy | 92.88% |
| Precision (positive class) | 94.02% |
| Recall (positive class) | 97.05% |
| F1-score (positive class) | 95.51% |
| ROC-AUC | 96.75% |

### Comparison with the Unigram Model

| Metric | Unigram TF-IDF | Bigram TF-IDF |
|---|---:|---:|
| Accuracy | 90.16% | 92.88% |
| Precision | 91.94% | 94.02% |
| Recall | 95.79% | 97.05% |
| F1-score | 93.82% | 95.51% |

### Interpretation

The bigram model improves all four reported classification metrics.

Accuracy increases by approximately 2.72 percentage points, while the F1-score increases by approximately 1.69 percentage points.

The improvement suggests that two-word combinations provide useful information for distinguishing sentiment.

In particular, phrases involving negation and descriptive expressions can be represented more effectively than when words are considered independently.

### Key Finding

Bigram TF-IDF provides the strongest traditional machine learning result so far and becomes the leading candidate for the final deployment model.

The next experiments investigate whether adjusting the classification threshold can improve the balance between precision and recall.

# 12. Understanding Prediction Probabilities

Logistic Regression can produce probability estimates in addition to direct class predictions.

For each review, the model estimates the probability of belonging to each sentiment class.

The positive-class probability is particularly useful because the default classification rule assigns a positive label when this probability is at least 0.50.

### Why Examine Probabilities?

A classification threshold determines how probability estimates are converted into final class predictions.

Changing this threshold can alter the number of false positives and false negatives.

This provides an opportunity to investigate whether the default threshold is appropriate for the project's classification objective.

In [28]:
y_proba = model1.predict_proba(X_test_tfidf_bigram)
y_proba[:5]

array([[0.0887424 , 0.9112576 ],
       [0.02633783, 0.97366217],
       [0.00490367, 0.99509633],
       [0.90571902, 0.09428098],
       [0.00212475, 0.99787525]])

## 13. Experimenting with a 0.60 Classification Threshold

The default binary classification threshold is 0.50.

In this experiment, the threshold is increased to 0.60.

A review is classified as positive only when its estimated positive-class probability is at least 0.60. Reviews below this threshold are classified as negative.

### Reason for This Experiment

A higher threshold makes the classifier more conservative when assigning the positive class.

This may reduce false positives, but it can also increase false negatives because some genuinely positive reviews may now be classified as negative.

The experiment therefore investigates the trade-off between precision and recall rather than assuming that a higher threshold will automatically improve performance.

In [29]:
positive_proba = y_proba[:, 1]
y_pred_06 = (positive_proba >= 0.6).astype(int)

In [30]:
classification_report(y_test, y_pred_06)

'              precision    recall  f1-score   support\n\n           0       0.84      0.84      0.84     24935\n           1       0.95      0.95      0.95     88756\n\n    accuracy                           0.93    113691\n   macro avg       0.90      0.90      0.90    113691\nweighted avg       0.93      0.93      0.93    113691\n'

### Interpreting the 0.60 Threshold Results

The 0.60 threshold produces:

| Metric | Score |
|---|---:|
| Accuracy | 92.87% |
| Precision | 95.38% |
| Recall | 95.49% |
| F1-score | 95.44% |

Compared with the default threshold, precision increases while recall decreases.

This is the expected effect of making positive predictions more conservative.

The model becomes less likely to label a negative review as positive, but it also misses more genuinely positive reviews.

The F1-score is approximately 95.44%, which is slightly lower than the default-threshold F1-score of 95.51%.

### Key Finding

The higher threshold improves precision but does not improve the overall F1-score.

Therefore, the default threshold remains preferable when the objective is to maximise the observed F1-score.

## 14. Decision Threshold Optimization

The classification threshold is selected using a validation split created only from the original training data.

The held-out test set is not used to choose the threshold. A temporary Bigram TF-IDF + Logistic Regression model is fitted on the threshold-training partition, and its validation predictions are used to select the threshold. The selected threshold is then transferred to the final Bigram TF-IDF model, which was trained on the complete original training set, and evaluated on the untouched test set.

Because the temporary and final models are fitted on different amounts of training data, the selected threshold is a validation-based estimate transferred to the final model; it is not guaranteed to be the exact optimal threshold for that final fitted model. The test set remains reserved for evaluation only.

### Why Use Positive-Class F1-Score?

The positive class is encoded as 1. Therefore, threshold selection uses `f1_score(..., pos_label=1)` and probabilities from `predict_proba()[:, 1]`.

The F1-score is the harmonic mean of positive-class precision and recall. It is useful when both false positives and false negatives matter.

A higher threshold generally makes positive predictions more conservative, while a lower threshold generally increases the number of positive predictions. The validation experiment measures this trade-off without tuning directly on the test set.


In [31]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np

# Create a validation split from the original training data only
X_threshold_train, X_threshold_val, y_threshold_train, y_threshold_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42
)

# Fit a fresh bigram vectorizer only on the threshold-training partition
threshold_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    max_features=50000
)

X_threshold_train_tfidf = threshold_vectorizer.fit_transform(X_threshold_train)
X_threshold_val_tfidf = threshold_vectorizer.transform(X_threshold_val)

# Train a temporary classifier for threshold selection
threshold_model = LogisticRegression(max_iter=500)
threshold_model.fit(X_threshold_train_tfidf, y_threshold_train)

# Positive-class probabilities on validation data (class label 1 = Positive)
validation_positive_proba = threshold_model.predict_proba(
    X_threshold_val_tfidf
)[:, 1]

# Select the threshold that maximizes positive-class F1 on validation data
thresholds = np.arange(0.10, 0.91, 0.01)
threshold_results = []

for threshold in thresholds:
    validation_predictions = (
        validation_positive_proba >= threshold
    ).astype(int)

    validation_f1 = f1_score(
        y_threshold_val,
        validation_predictions,
        pos_label=1
    )

    threshold_results.append((threshold, validation_f1))

best_threshold, best_validation_f1 = max(
    threshold_results,
    key=lambda item: item[1]
)

print("Selected threshold:", round(float(best_threshold), 2))
print("Validation positive-class F1:", round(float(best_validation_f1), 4))


Selected threshold: 0.52
Validation positive-class F1: 0.9538


In [32]:
# Evaluate the validation-selected threshold on the held-out test set.
# model1 and X_test_tfidf_bigram are the final bigram model and test features.

test_positive_proba = model1.predict_proba(
    X_test_tfidf_bigram
)[:, 1]

y_pred_best = (
    test_positive_proba >= best_threshold
).astype(int)

print("Validation-selected threshold:", round(float(best_threshold), 2))
print("\nTest Classification Report:")
print(classification_report(y_test, y_pred_best))

print("Test Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("Test positive-class F1:",
      f1_score(y_test, y_pred_best, pos_label=1))


Validation-selected threshold: 0.52

Test Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.79      0.83     24935
           1       0.94      0.97      0.96     88756

    accuracy                           0.93    113691
   macro avg       0.91      0.88      0.89    113691
weighted avg       0.93      0.93      0.93    113691

Test Confusion Matrix:
[[19744  5191]
 [ 2853 85903]]
Test positive-class F1: 0.9552738393105366


### Threshold Optimization: Results and Interpretation

The threshold is selected by maximizing positive-class F1-score on a validation partition drawn from the training data. The held-out test set is used only after the threshold has been selected.

The threshold-selection code cell prints the validation F1-score used to select the threshold. The following test-evaluation code cell applies that selected threshold to the final Bigram TF-IDF model and prints the test classification report and confusion matrix.

### Full-Test-Set Results

The default threshold of 0.50 and the validation-selected threshold of 0.52 produce the following results on the same held-out test set:

| Metric | Default threshold (0.50) | Validation-selected threshold (0.52) |
|---|---:|---:|
| Accuracy | 92.88% | 92.92% |
| Positive-class F1-score | 95.51% | 95.53% |

The change is marginal: accuracy increases by **0.04 percentage points**, and positive-class F1-score increases by **0.02 percentage points**.

### Interpretation

Threshold selection is a model decision made using validation data. The test result must not be used to choose a different threshold or repeatedly tune the model.

The selected threshold is obtained using a temporary model fitted on the threshold-training partition and transferred to the final model fitted on the complete original training set. Since these are different fitted models, the transferred threshold is an estimate rather than a threshold independently optimized for the final fitted model.

The final deployment model remains the Bigram TF-IDF + Logistic Regression model. This threshold experiment does not automatically change the threshold used by the deployed application.


# 15. Exploring GloVe Word Embeddings

TF-IDF represents words according to their statistical importance within the document collection.

However, it does not directly encode semantic relationships between words.

To explore an alternative representation, this project introduces GloVe (Global Vectors for Word Representation).

GloVe is a pretrained word embedding technique that learns dense vector representations from global word co-occurrence statistics.

### Why Explore GloVe?

Words with related meanings may have similar vector representations.

For example, words such as `excellent`, `great`, and `fantastic` may occupy nearby regions in the embedding space.

This experiment investigates whether semantic word representations can provide useful information for sentiment classification beyond TF-IDF features.

The selected pretrained model is `glove-wiki-gigaword-100`, which represents each word using a 100-dimensional vector.

In [33]:
model = api.load("glove-wiki-gigaword-100")

[==================================================] 100.0% 128.1/128.1MB downloaded


In [34]:
model["excellent"].shape

(100,)

In [35]:
model.most_similar("excellent", topn=10)

[('good', 0.7936238050460815),
 ('quality', 0.7606268525123596),
 ('terrific', 0.7415624260902405),
 ('superb', 0.740295946598053),
 ('best', 0.7287570238113403),
 ('impressive', 0.721752405166626),
 ('better', 0.7102464437484741),
 ('solid', 0.6999199986457825),
 ('decent', 0.6907013058662415),
 ('perfect', 0.6837304830551147)]

In [36]:
model.similarity("excellent", "fantastic")

np.float32(0.5713639)

In [37]:
model.similarity("excellent", "terrible")

np.float32(0.31886587)

In [38]:
review = X_train.iloc[0]

words = review.lower().split()

word_vectors = [
    model[word]
    for word in words
    if word in model
]

review_vector = np.mean(word_vectors, axis=0)

print("Review:")
print(review)

print("\nNumber of words:", len(words))
print("Words with GloVe vectors:", len(word_vectors))
print("Review vector shape:", review_vector.shape)
print("\nFirst 10 values:")
print(review_vector[:10])

Review:
We have purchased these several times.  This past time our dog got it out of the holder and ate the whole thing in less than an hour.  Too expensive to have it last this short of time.

Number of words: 37
Words with GloVe vectors: 34
Review vector shape: (100,)

First 10 values:
[-0.09919062  0.24226707  0.40343326 -0.2890118  -0.12304228  0.13684636
 -0.10542948  0.24128589 -0.18182482 -0.01051159]


In [39]:
def review_to_vector(review):
    words = review.lower().split()

    word_vectors = [
        model[word]
        for word in words
        if word in model
    ]

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)
review_vector = review_to_vector(X_train.iloc[0])

print("Shape:", review_vector.shape)
print("First 10 values:", review_vector[:10])

Shape: (100,)
First 10 values: [-0.09919062  0.24226707  0.40343326 -0.2890118  -0.12304228  0.13684636
 -0.10542948  0.24128589 -0.18182482 -0.01051159]


In [40]:
X_train_glove = np.array([
    review_to_vector(review)
    for review in X_train
])

print("X_train_glove shape:", X_train_glove.shape)

X_train_glove shape: (454763, 100)


In [41]:
X_test_glove = np.array([
    review_to_vector(review)
    for review in X_test
])

print("X_test_glove shape:", X_test_glove.shape)

X_test_glove shape: (113691, 100)


## 16. Logistic Regression with GloVe Features

Train Logistic Regression using the fixed-length review representations generated from GloVe embeddings.

The classification algorithm is kept consistent with the earlier experiments to make the comparison between different text representations easier.

The model learns a decision boundary using the dense 100-dimensional review vectors.

Its predictions are evaluated using the held-out test data.

In [42]:
glove_model = LogisticRegression(max_iter=1000)
glove_model.fit(X_train_glove, y_train)

LogisticRegression(max_iter=1000)

In [43]:
y_pred_glove = glove_model.predict(X_test_glove)
print(accuracy_score(y_test, y_pred_glove))
print(precision_score(y_test, y_pred_glove))
print(recall_score(y_test, y_pred_glove))
print(classification_report(y_test, y_pred_glove))
print(confusion_matrix(y_test, y_pred_glove))

0.8139870350335559
0.8356168463692143
0.948273919509667
              precision    recall  f1-score   support

           0       0.65      0.34      0.44     24935
           1       0.84      0.95      0.89     88756

    accuracy                           0.81    113691
   macro avg       0.74      0.64      0.67    113691
weighted avg       0.79      0.81      0.79    113691

[[ 8378 16557]
 [ 4591 84165]]


### GloVe Model: Results and Interpretation

The GloVe-based Logistic Regression model achieves:

| Metric | Score |
|---|---:|
| Accuracy | 81.40% |
| Precision | 83.56% |
| Recall | 94.83% |
| F1-score | 88.84% |

### Interpretation

The model achieves relatively high positive-class recall, indicating that it identifies most positive reviews.

However, its accuracy and F1-score are considerably lower than those of the TF-IDF models.

This suggests that the simple review-level combination of GloVe word vectors does not preserve enough information to match the performance of the sparse TF-IDF representation in this experiment.

One possible reason is that averaging word embeddings loses word order and may weaken the representation of phrases involving negation or mixed sentiment.

### Key Finding

Pretrained semantic word embeddings are useful for representing relationships between words, but a simple averaged review representation is not necessarily more effective than TF-IDF for this classification task.

The result motivates an additional experiment using sentence-level embeddings.

# 17. Sentence Transformer Embeddings

The next experiment uses Sentence Transformers to generate dense representations of complete sentences and reviews.

The selected pretrained model is `all-MiniLM-L6-v2`.

Unlike a simple average of static word embeddings, Sentence Transformers are designed to produce sentence-level representations that capture broader semantic information.

### Why Sentence Transformers?

A review expresses an overall opinion through multiple words and phrases.

A sentence-level embedding provides a representation of the complete text rather than treating each word as an independent feature.

The selected model generates 384-dimensional embeddings.

These representations are then used as input features for a Logistic Regression classifier.

In [44]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 6.1 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [45]:
from sentence_transformers import SentenceTransformer

In [46]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [47]:
X_sample_sentence = sentence_model.encode(
    X_train.iloc[:100].tolist(),
    show_progress_bar=False
)

print("Shape:", X_sample_sentence.shape)

Shape: (100, 384)


In [48]:
sentences = [
    "The food was excellent.",
    "The food was terrible."
]

embeddings = sentence_model.encode(
    sentences,
    show_progress_bar=False
)

print("Shape:", embeddings.shape)

Shape: (2, 384)


In [49]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    embeddings[0].reshape(1, -1),
    embeddings[1].reshape(1, -1)
)

print("Cosine similarity:", similarity[0][0])

Cosine similarity: 0.60678697


In [50]:
sentences_2 = [
    "The food was excellent.",
    "The food was fantastic."
]

embeddings_2 = sentence_model.encode(
    sentences_2,
    show_progress_bar=False
)

similarity_2 = cosine_similarity(
    embeddings_2[0].reshape(1, -1),
    embeddings_2[1].reshape(1, -1)
)

print("Cosine similarity:", similarity_2[0][0])

Cosine similarity: 0.9456593


### Interpreting Sentence Similarity

The similarity between:

- "The food was excellent."
- "The food was fantastic."

is approximately **0.946**.

The similarity between:

- "The food was excellent."
- "The food was terrible."

is approximately **0.607**.

The semantically similar positive sentences receive a higher similarity score than the contrasting positive and negative sentences.

This provides an intuitive demonstration that the model captures some meaningful semantic relationships.

However, the second pair still has a positive similarity value because the sentences share much of their vocabulary and subject matter.

### Key Finding

Sentence embeddings encode semantic information beyond individual word identity, but similarity scores alone do not establish whether the representation will produce a strong sentiment classifier.

A supervised classification experiment is required to evaluate that.

In [51]:
X_train_sample = X_train.iloc[:10000].tolist()

X_train_sentence = sentence_model.encode(
    X_train_sample,
    show_progress_bar=True,
    batch_size=32
)

print("Shape:", X_train_sentence.shape)

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Shape: (10000, 384)


In [52]:
y_train_sample = y_train.iloc[:10000]

sentence_model_lr = LogisticRegression(
    max_iter=500,
    random_state=42
)

sentence_model_lr.fit(
    X_train_sentence,
    y_train_sample
)

y_train_pred_sentence = sentence_model_lr.predict(X_train_sentence)

print("Training Accuracy:", accuracy_score(y_train_sample, y_train_pred_sentence))
print("\nClassification Report:")
print(classification_report(y_train_sample, y_train_pred_sentence))

print("\nConfusion Matrix:")
print(confusion_matrix(y_train_sample, y_train_pred_sentence))

# Take a manageable test subset
X_test_sample = X_test.iloc[:2000].tolist()
y_test_sample = y_test.iloc[:2000]

# Convert test reviews into sentence embeddings
X_test_sentence = sentence_model.encode(
    X_test_sample,
    show_progress_bar=True,
    batch_size=32
)

print("Shape:", X_test_sentence.shape)

Training Accuracy: 0.8592

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.48      0.59      2156
           1       0.87      0.96      0.91      7844

    accuracy                           0.86     10000
   macro avg       0.83      0.72      0.75     10000
weighted avg       0.85      0.86      0.85     10000


Confusion Matrix:
[[1028 1128]
 [ 280 7564]]


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Shape: (2000, 384)


In [53]:
y_test_pred_sentence = sentence_model_lr.predict(X_test_sentence)

print("Test Accuracy:", accuracy_score(y_test_sample, y_test_pred_sentence))

print("\nClassification Report:")
print(classification_report(y_test_sample, y_test_pred_sentence))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_sample, y_test_pred_sentence))

Test Accuracy: 0.833

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.41      0.52       442
           1       0.85      0.95      0.90      1558

    accuracy                           0.83      2000
   macro avg       0.78      0.68      0.71      2000
weighted avg       0.82      0.83      0.82      2000


Confusion Matrix:
[[ 183  259]
 [  75 1483]]


### Sentence Transformer: Results and Interpretation

On the 2,000-review test subset, the Sentence Transformer-based classifier achieves:

| Metric | Score |
|---|---:|
| Accuracy | 83.30% |
| Positive-class Precision | 85.13% |
| Positive-class Recall | 95.19% |
| Positive-class F1-score | 89.88% |

The confusion matrix is:

| | Predicted Negative | Predicted Positive |
|---|---:|---:|
| Actual Negative | 183 | 259 |
| Actual Positive | 75 | 1,483 |

### Interpretation

The model identifies most positive reviews, achieving approximately 95.19% positive-class recall.

However, negative-class recall is approximately 41.40%. Out of 442 negative reviews in the evaluated subset, only 183 are correctly identified.

The classifier therefore has difficulty distinguishing negative reviews from positive reviews, despite using sentence-level semantic representations.

### Key Finding

Sentence-level embeddings capture semantic similarity, but this experiment does not demonstrate better sentiment classification performance than the full-test TF-IDF models.

The evaluation subset is smaller and has a different class composition, so the numerical results should not be treated as a strict head-to-head comparison.

The next experiment investigates BERT, which uses contextual token representations.


In [54]:
# Create a small error-analysis table

import pandas as pd

error_analysis_sentence = pd.DataFrame({
    "Review": X_test.iloc[:2000].values,
    "Actual": y_test_sample.values,
    "Predicted": y_test_pred_sentence
})

# Keep only incorrectly classified reviews
errors_sentence = error_analysis_sentence[
    error_analysis_sentence["Actual"] != error_analysis_sentence["Predicted"]
]

print("Total errors:", len(errors_sentence))

# Show 20 examples
pd.set_option("display.max_colwidth", 300)

errors_sentence.head(20)

Total errors: 334


,Review,Actual,Predicted
3,"I bought this food for my dog (who has very sensitive skin), and, maybe either a) it is not for him, or, b), I didn't introduce it gradually enough, but, he did develop some vomiting and loose stools when eating (fed him only for about 3 days). It sounds like fabulous food, with quality ingredi...",0,1
13,"The consistency is a little weird due to the types of grains. And I must ask,why in the world would each cup be equal to 2 servings? It's a waste for those who are not doubling up on servings. It takes 5 minutes to prepare at which time the oatmeal is barely warm and needs to be reheated. I ex...",0,1
15,Firm olives but a little too salty for my taste. I know these olives are supposed to be salty but they should not be a salt lick. Good when eaten with other food.,0,1
19,"Coffee is ok but don't know whether I will reorder or not. Rather weak but if your machine has the water level setting, I would suggest that you use one of the smaller cups.",0,1
20,I used to buy these from Amazon at least 2-3 times a month. The last 2 shipments were hard as a rock and the potatoes looked like plastic. I definitely did not give these to my shih tzu. The treats used to be softer and you could tear them into smaller pieces. The last ones I received I could...,0,1
21,This mix makes muffins that are very dry. If you add to it you can make the muffins a little more moist and tasty.,0,1
23,"I like Kraft Macaroni and Cheese and hoped this would be a convenient and easy alternative to the box meal. It is relatively quick and easy to prepare, but it tastes excessively salty.",0,1
25,this product contains gelatin and other ingredients. tastes great and my baby likes it but I prefer to feed her foods with less additives.,0,1
33,"Excellent vanilla flavor. However I was not ""delighted"" that it curdled in hot coffee!!! Nasty and very unappetizing in appearance. I gave up using for hot coffee and use it for ""iced coffee"" with great results.",0,1
38,I had been feeding my cats dry Go! cat food. Since I had opened the last bag of Go my cats had refused to eat. Maybe it was bad. They had never liked Go very much. One cat was looking slightly skeletal. When I opened this bag and a whiff of smell escaped they charged me. They absolutely ad...,1,0


### Sentence Transformer Error Analysis

Inspect the misclassified reviews from the 2,000-review evaluation subset.

The classifier produces 334 errors.

The confusion matrix indicates that many of these errors are negative reviews predicted as positive.

Reviewing individual examples helps investigate whether the model struggles with:

- Mixed positive and negative opinions.
- Product-specific complaints.
- Negation.
- Reviews that describe an experience without using strongly negative words.
- Long reviews containing both praise and criticism.

This qualitative analysis complements the numerical metrics and helps explain the model's weaker negative-class performance.

# 18. Exploring BERT for Sentiment Analysis

BERT (Bidirectional Encoder Representations from Transformers) is a pretrained language model that generates contextual representations of text.

Unlike static word embeddings, BERT represents a word based on the surrounding words in the input sequence.

For example, the representation of a word can change depending on whether it appears in a positive or negative sentence.

### Why Explore BERT?

The previous experiments investigated:

- TF-IDF, which represents word and phrase importance.
- GloVe, which provides pretrained static word vectors.
- Sentence Transformers, which generate sentence-level semantic embeddings.

BERT provides another approach through contextual transformer representations.

### Experiments Performed

This project uses BERT in two separate experiments:

1. **Pretrained BERT representations:** Load `bert-base-uncased`, extract contextual token and sentence representations, and examine semantic similarity between example sentences and embeddings from sample Amazon reviews. These extracted embeddings are explored only; they are not used to train a separate traditional classifier.
2. **Fine-tuned BERT classifier:** Load `bert-base-uncased` with a sequence-classification head and fine-tune it on labelled Amazon reviews for binary sentiment classification.

The classification results reported in the following section belong to the fine-tuned BERT model, not to a traditional classifier trained on the pretrained embeddings.


In [55]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load pretrained BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

print("BERT loaded successfully")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT loaded successfully


In [56]:
import torch
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The food was great.",
    "The food was not great.",
    "The food was fantastic."
]

# Tokenize
inputs = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Get BERT representations
with torch.no_grad():
    outputs = bert_model(**inputs)

# Last hidden state:
# (number of sentences, number of tokens, 768)
last_hidden = outputs.last_hidden_state

print("Shape:", last_hidden.shape)

# Simple sentence representation:
# average the token embeddings for each sentence
attention_mask = inputs["attention_mask"].unsqueeze(-1)

masked_embeddings = last_hidden * attention_mask
sentence_embeddings = masked_embeddings.sum(dim=1) / attention_mask.sum(dim=1)

print("Sentence embedding shape:", sentence_embeddings.shape)

# Similarity between the sentences
similarity_matrix = cosine_similarity(
    sentence_embeddings.numpy()
)

print("\nCosine Similarity:")
print(similarity_matrix)

Shape: torch.Size([3, 8, 768])
Sentence embedding shape: torch.Size([3, 768])

Cosine Similarity:
[[0.99999976 0.9096974  0.97819775]
 [0.9096974  0.99999976 0.88108027]
 [0.97819775 0.88108027 1.0000001 ]]


In [57]:
# Test BERT on a small batch of actual Amazon reviews

sample_reviews = X_train.iloc[:5].tolist()

inputs = tokenizer(
    sample_reviews,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = bert_model(**inputs)

# [CLS] token representation
cls_embeddings = outputs.last_hidden_state[:, 0, :]

print("Number of reviews:", len(sample_reviews))
print("BERT embedding shape:", cls_embeddings.shape)

Number of reviews: 5
BERT embedding shape: torch.Size([5, 768])


In [58]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/reetdubey/reviews/Reviews.csv"
)

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (568454, 10)
   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary  \
0  Good Quality Dog Food   
1      Not as Advertised   
2  "Delight" says it all   
3         Cough Me

In [59]:
from sklearn.model_selection import train_test_split

df["Sentiment"] = df["Score"].map({
    1: 0,
    2: 0,
    3: 0,
    4: 1,
    5: 1
})

X = df["Text"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training labels:")
print(y_train.value_counts())

Training samples: 454763
Testing samples: 113691
Training labels:
Sentiment
1    355021
0     99742
Name: count, dtype: int64


In [60]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

bert_classifier = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Tokenizer loaded successfully")
print("BERT classifier loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer loaded successfully
BERT classifier loaded successfully


In [61]:
from sklearn.model_selection import train_test_split

X_bert_train, X_bert_val, y_bert_train, y_bert_val = train_test_split(
    X_train,
    y_train,
    train_size=5000,
    test_size=1000,
    stratify=y_train,
    random_state=42
)

print("BERT training samples:", len(X_bert_train))
print("BERT validation samples:", len(X_bert_val))

BERT training samples: 5000
BERT validation samples: 1000


In [62]:
import torch
from torch.utils.data import Dataset, DataLoader

X_train_ft = X_bert_train.tolist()
y_train_ft = y_bert_train.tolist()

X_val_ft = X_bert_val.tolist()
y_val_ft = y_bert_val.tolist()


class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            key: value.squeeze(0)
            for key, value in encoding.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item


train_dataset = ReviewDataset(
    X_train_ft, y_train_ft, tokenizer
)

val_dataset = ReviewDataset(
    X_val_ft, y_val_ft, tokenizer
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Training samples: 5000
Validation samples: 1000


In [63]:
from torch.utils.data import DataLoader

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 313
Validation batches: 63


In [64]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_classifier.to(device)

learning_rate = 2e-5
epochs = 1

optimizer = torch.optim.AdamW(
    bert_classifier.parameters(),
    lr=learning_rate
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("Learning rate:", learning_rate)
print("Epochs:", epochs)
print("Optimizer: AdamW")

Device: cuda
GPU: Tesla T4
Learning rate: 2e-05
Epochs: 1
Optimizer: AdamW


In [65]:
from tqdm.auto import tqdm

bert_classifier.train()

total_loss = 0

for batch in tqdm(train_loader, desc="Fine-tuning BERT"):

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()

    outputs = bert_classifier(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

average_train_loss = total_loss / len(train_loader)

print("Average Training Loss:", average_train_loss)

Fine-tuning BERT:   0%|          | 0/313 [00:00<?, ?it/s]

Average Training Loss: 0.3038011513317165


In [66]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

bert_classifier.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating BERT"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = bert_classifier(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Validation Accuracy:", accuracy_score(all_labels, all_predictions))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_predictions,
    target_names=["Negative", "Positive"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

Evaluating BERT:   0%|          | 0/63 [00:00<?, ?it/s]

Validation Accuracy: 0.903

Classification Report:
              precision    recall  f1-score   support

    Negative       0.84      0.69      0.76       219
    Positive       0.92      0.96      0.94       781

    accuracy                           0.90      1000
   macro avg       0.88      0.83      0.85      1000
weighted avg       0.90      0.90      0.90      1000


Confusion Matrix:
[[151  68]
 [ 29 752]]


### Fine-Tuned BERT: Results and Interpretation

The fine-tuned BERT model achieves the following results on the 1,000-review validation subset:

| Metric | Score |
|---|---:|
| Accuracy | 90.30% |
| Negative Precision | 84.00% |
| Negative Recall | 69.00% |
| Negative F1-score | 76.00% |
| Positive Precision | 92.00% |
| Positive Recall | 96.00% |
| Positive F1-score | 94.00% |

The confusion matrix is:

| | Predicted Negative | Predicted Positive |
|---|---:|---:|
| Actual Negative | 151 | 68 |
| Actual Positive | 29 | 752 |

### Interpretation

The model correctly classifies **903 out of 1,000 validation reviews**, achieving an accuracy of 90.30%.

- It correctly identifies 151 of the 219 negative reviews.
- It correctly identifies 752 of the 781 positive reviews.
- Negative-class recall is 69%, indicating that the model identifies approximately 69% of actual negative reviews.
- Positive-class recall is 96%, showing that the model identifies most positive reviews.
- The model incorrectly classifies 68 negative reviews as positive and 29 positive reviews as negative.

These results indicate that the fine-tuned BERT model performs better at identifying positive reviews than negative reviews on this validation subset.

### Important Evaluation Limitation

This BERT result is based on a 1,000-review validation subset, whereas the main TF-IDF results are based on the full 113,691-review test set.

Therefore, the 90.30% BERT accuracy should not be interpreted as a directly comparable result against the 92.88% bigram TF-IDF accuracy.

The different sample sizes and evaluation partitions prevent a controlled head-to-head comparison.

### Key Finding

Fine-tuning BERT successfully produces a contextual sentiment classifier. However, this particular experiment does not establish that BERT outperforms the selected Bigram TF-IDF model.

The experiment uses 5,000 training reviews and one training epoch, and requires substantially more computational resources than the TF-IDF approach.
### Interpretation

The model correctly classifies 893 out of 1,000 validation reviews.

It identifies 185 of the 219 negative reviews and 708 of the 781 positive reviews.

The negative-class recall is approximately 84%, while positive-class recall is approximately 91%. The model identifies a larger proportion of positive reviews, but it also detects a substantial proportion of negative reviews.

### Important Evaluation Limitation

This BERT result is based on a 1,000-review validation subset, whereas the main TF-IDF results are based on the full 113,691-review test set.

Therefore, the 89.30% BERT accuracy should not be interpreted as a directly comparable result against the 92.88% bigram TF-IDF accuracy.

The different sample sizes and evaluation partitions prevent a controlled head-to-head comparison.

### Key Finding

Fine-tuning BERT successfully produces a contextual sentiment classifier, but this particular experiment does not demonstrate superior performance to the final TF-IDF model.

It also requires substantially more computation.


In [67]:
bert_classifier.save_pretrained("/kaggle/working/bert_sentiment_model")
tokenizer.save_pretrained("/kaggle/working/bert_sentiment_model")

print("BERT model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT model and tokenizer saved successfully!


In [68]:
import os

model_path = "/kaggle/working/bert_sentiment_model"

print("Saved files:")
for file in os.listdir(model_path):
    print(file)

Saved files:
model.safetensors
tokenizer_config.json
config.json
tokenizer.json


In [69]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

bert_classifier.eval()

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = bert_classifier(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_true_labels.extend(batch["labels"].numpy())

bert_results = pd.DataFrame({
    "Review": X_bert_val.reset_index(drop=True),
    "Actual": all_true_labels,
    "Predicted": all_predictions
})

bert_results["Actual_Label"] = bert_results["Actual"].map({
    0: "Negative", 1: "Positive"
})

bert_results["Predicted_Label"] = bert_results["Predicted"].map({
    0: "Negative", 1: "Positive"
})

bert_results["Correct"] = (
    bert_results["Actual"] == bert_results["Predicted"]
)

print("Total predictions:", len(bert_results))
print("Correct predictions:", bert_results["Correct"].sum())
print("Misclassified reviews:", (~bert_results["Correct"]).sum())

Total predictions: 1000
Correct predictions: 903
Misclassified reviews: 97


In [70]:
# Run this cell after the BERT prediction/error-analysis cell that creates bert_results.
# Labels: 0 = Negative, 1 = Positive

# False Positive: actual Negative (0), predicted Positive (1)
false_positives = bert_results[
    (bert_results["Actual"] == 0) &
    (bert_results["Predicted"] == 1)
]

# False Negative: actual Positive (1), predicted Negative (0)
false_negatives = bert_results[
    (bert_results["Actual"] == 1) &
    (bert_results["Predicted"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

print("\nFALSE POSITIVE EXAMPLES:")
for review in false_positives["Review"].head(5):
    print("-", review[:500], "\n")

print("\nFALSE NEGATIVE EXAMPLES:")
for review in false_negatives["Review"].head(5):
    print("-", review[:500], "\n")

False Positives: 68
False Negatives: 29

FALSE POSITIVE EXAMPLES:
- This is a sweet smelling product. However, the doggies can't stand it when they are sprayed. If I can get to them, ONE spray is sufficient! 

- IN A NUTSHELL: FEW NATURAL FOODS COULD BE WORSE FOR YOU THAN LARD!<br /><br />WHAT IS WORSE?  --- Hydrogenated LARD, which is loaded with TRANS FATTY ACIDS - CHOLESTEROL - SATURATED FAT - AGRICULTURAL CHEMICALS - HORMONES ETC. AND that's what this product is.<br /><br />SOME INTERESTING STUFF: WHERE IS THE LARD ON THEIR WEBSITE?<br /><br />Dial, the soap people, own Armour, and Armour makes this Hydrogenated, "All Natural" Lard. However, though Armour owns up to making many less than healthy edi 

- I got this for my dog during the heat of summer since she loves eating ice so much! She likes this toy, although she gets bored with it very easily (I guess it's the crunch factor of ice she's missing) so it normally sits on the floor until it melts. Word to the wise, don't let your

In [71]:
bert_classifier.eval()

all_probabilities = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = bert_classifier(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(outputs.logits, dim=1)

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )

bert_results["Negative_Probability"] = [
    p[0] for p in all_probabilities
]

bert_results["Positive_Probability"] = [
    p[1] for p in all_probabilities
]

bert_results["Confidence"] = [
    max(p) for p in all_probabilities
]

print("Average confidence on correct predictions:",
      bert_results.loc[bert_results["Correct"], "Confidence"].mean())

print("Average confidence on incorrect predictions:",
      bert_results.loc[~bert_results["Correct"], "Confidence"].mean())

Average confidence on correct predictions: 0.907207
Average confidence on incorrect predictions: 0.7201393


In [72]:
thresholds = [0.60, 0.70, 0.80, 0.90]

for threshold in thresholds:
    flagged = bert_results[
        bert_results["Confidence"] < threshold
    ]

    errors = flagged[~flagged["Correct"]]

    print(f"\nConfidence threshold: {threshold}")
    print("Reviews flagged:", len(flagged))
    print("Incorrect predictions among flagged:", len(errors))
    print(
        "Error percentage among flagged:",
        round(len(errors) / len(flagged) * 100, 2)
        if len(flagged) > 0 else 0
    )


Confidence threshold: 0.6
Reviews flagged: 58
Incorrect predictions among flagged: 28
Error percentage among flagged: 48.28

Confidence threshold: 0.7
Reviews flagged: 128
Incorrect predictions among flagged: 44
Error percentage among flagged: 34.38

Confidence threshold: 0.8
Reviews flagged: 221
Incorrect predictions among flagged: 66
Error percentage among flagged: 29.86

Confidence threshold: 0.9
Reviews flagged: 349
Incorrect predictions among flagged: 85
Error percentage among flagged: 24.36


In [73]:
total_errors = (~bert_results["Correct"]).sum()

for threshold in [0.60, 0.70, 0.80, 0.90]:
    flagged = bert_results[
        bert_results["Confidence"] < threshold
    ]

    errors_captured = (~flagged["Correct"]).sum()

    print(
        f"Threshold {threshold}: "
        f"{errors_captured}/{total_errors} errors captured "
        f"({errors_captured / total_errors * 100:.2f}%)"
    )

Threshold 0.6: 28/97 errors captured (28.87%)
Threshold 0.7: 44/97 errors captured (45.36%)
Threshold 0.8: 66/97 errors captured (68.04%)
Threshold 0.9: 85/97 errors captured (87.63%)


In [74]:
bert_results["Review_Status"] = np.where(
    bert_results["Confidence"] < 0.80,
    "Needs Manual Review",
    "High Confidence"
)

print(bert_results["Review_Status"].value_counts())

print("\nExamples requiring manual review:")

display(
    bert_results[
        bert_results["Review_Status"] == "Needs Manual Review"
    ][
        ["Review", "Actual_Label", "Predicted_Label", "Confidence", "Correct"]
    ].head(10)
)

Review_Status
High Confidence        779
Needs Manual Review    221
Name: count, dtype: int64

Examples requiring manual review:


,Review,Actual_Label,Predicted_Label,Confidence,Correct
4,"What I liked best was the flavor, both for spearmint and cinnamon. However, I discovered I had a problem digesting xylitol (as some do), so won't be purchasing additional items made with it.",Negative,Negative,0.528542,True
5,"The Clear Men Scalp Therapy 2-in-1 Anti-Dandruff Shampoo and Conditioner is a fine choice for men who need a dandruff shampoo that works. This product contains ginseng, mint, and tea tree oil along with its active ingredient, pyrithione zinc, which means it may sting a little bit until it's rin...",Positive,Positive,0.776108,True
7,They only like this brand and flavor of treat. Won't go for the soft version. One of my cats will actually stick his paw down into the canister and drag a few up the side if I hold the can within reach. I find this product reasonably priced at Target.,Positive,Positive,0.784061,True
13,"These peanuts are rock hard. I ate a handful of peanuts and got a bad stomach ache. They don't taste bad, and not salty at all. I don't understand why they are so hard. Ouch. I love the peanuts from Costco. They are also extra large and delicious but they never hurt my stomach. Must go bac...",Negative,Negative,0.572548,True
14,"IN A NUTSHELL: FEW NATURAL FOODS COULD BE WORSE FOR YOU THAN LARD!<br /><br />WHAT IS WORSE? --- Hydrogenated LARD, which is loaded with TRANS FATTY ACIDS - CHOLESTEROL - SATURATED FAT - AGRICULTURAL CHEMICALS - HORMONES ETC. AND that's what this product is.<br /><br />SOME INTERESTING STUFF: W...",Negative,Positive,0.558074,False
15,"I have been using a standard XLR to USB cable with my Shure 58 dynamic mic, which works okay, except for an annoying hissing noise that accompanies the majority of my vocal recordings. I bought the Icicle in hopes of eliminating those noise issues.<br /><br />When I received the Icicle, I could...",Negative,Negative,0.777918,True
17,"I have been using Spectrums brand for a year now. I use it for hair, cooking, oil pulling, as a body moisturizer, and as a air fresher in my oil burner. Recently I made the mistake of trying other brands. I will never do that again. I'm from the islands so I know the true quality of unrefined co...",Positive,Negative,0.641507,False
19,Seems like this company/brand is falling apart. They used to be great but they're just okay now. These used to be thin long strips that actually looked like achilles tendons (two points of insertion at 1 end and one point of attachment on the other end) as pictured. The new one is just a giant s...,Negative,Negative,0.718583,True
23,YOU SEND ONE FIRST THAT EXPIRATION DATE WAS 2005 --THE PACKAGE WAS APART WITH DUST ON IT!!!! I WILL KEEP YOU IN MY PRAYERS,Negative,Negative,0.734100,True
30,"I got this for my dog during the heat of summer since she loves eating ice so much! She likes this toy, although she gets bored with it very easily (I guess it's the crunch factor of ice she's missing) so it normally sits on the floor until it melts. Word to the wise, don't let your dog have thi...",Negative,Positive,0.645162,False


In [75]:
bert_results.to_csv(
    "/kaggle/working/bert_error_analysis.csv",
    index=False
)

print("Error analysis saved successfully!")

Error analysis saved successfully!


In [76]:
import os
import joblib

SAVE_DIR = "/kaggle/working/amazon_sentiment_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the additional model experiments.
# The final Bigram TF-IDF deployment pair is saved separately in the
# dedicated deployment-artifact section below.

# GloVe + Logistic Regression
joblib.dump(
    glove_model,
    f"{SAVE_DIR}/glove_logistic_regression.pkl"
)

# Sentence Transformer and its Logistic Regression classifier
sentence_model.save(f"{SAVE_DIR}/sentence_transformer")

joblib.dump(
    sentence_model_lr,
    f"{SAVE_DIR}/sentence_transformer_logistic_regression.pkl"
)

# Pretrained GloVe embeddings
model.save(f"{SAVE_DIR}/glove_embeddings.kv")

# Fine-tuned BERT
bert_dir = f"{SAVE_DIR}/bert_sentiment_model"
os.makedirs(bert_dir, exist_ok=True)

bert_classifier.save_pretrained(bert_dir)
tokenizer.save_pretrained(bert_dir)

print("Additional model artefacts saved successfully.")

for root, dirs, files in os.walk(SAVE_DIR):
    for file in files:
        print(os.path.join(root, file))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Additional model artefacts saved successfully.
/kaggle/working/amazon_sentiment_models/glove_embeddings.kv
/kaggle/working/amazon_sentiment_models/glove_embeddings.kv.vectors.npy
/kaggle/working/amazon_sentiment_models/glove_logistic_regression.pkl
/kaggle/working/amazon_sentiment_models/sentence_transformer_logistic_regression.pkl
/kaggle/working/amazon_sentiment_models/sentence_transformer/model.safetensors
/kaggle/working/amazon_sentiment_models/sentence_transformer/modules.json
/kaggle/working/amazon_sentiment_models/sentence_transformer/config_sentence_transformers.json
/kaggle/working/amazon_sentiment_models/sentence_transformer/tokenizer_config.json
/kaggle/working/amazon_sentiment_models/sentence_transformer/README.md
/kaggle/working/amazon_sentiment_models/sentence_transformer/config.json
/kaggle/working/amazon_sentiment_models/sentence_transformer/tokenizer.json
/kaggle/working/amazon_sentiment_models/sentence_transformer/sentence_bert_config.json
/kaggle/working/amazon_senti

In [77]:
print("Final deployment TF-IDF configuration:")
print("N-gram range:", tfidf_bigram.ngram_range)
print("Maximum features:", tfidf_bigram.max_features)

print("\nFinal deployment Logistic Regression:")
print("Model:", model1)
print("Classes:", model1.classes_)
print("Classifier feature count:", model1.n_features_in_)
print(
    "Vectorizer feature count:",
    len(tfidf_bigram.get_feature_names_out())
)


Final deployment TF-IDF configuration:
N-gram range: (1, 2)
Maximum features: 50000

Final deployment Logistic Regression:
Model: LogisticRegression(max_iter=500)
Classes: [0 1]
Classifier feature count: 50000
Vectorizer feature count: 50000


In [78]:
print("model1:", model1)
print("threshold_model:", threshold_model)

print("\nBigram feature shape:", X_train_tfidf_bigram.shape)
print("Unigram feature shape:", X_train_tfidf.shape)

model1: LogisticRegression(max_iter=500)
threshold_model: LogisticRegression(max_iter=500)

Bigram feature shape: (454763, 50000)
Unigram feature shape: (454763, 20000)


In [79]:
for name, clf in [
    ("model1", model1),
    ("threshold_model", threshold_model),
    ("glove_model", glove_model),
    ("sentence_model_lr", sentence_model_lr)
]:
    print(
        name,
        "| fitted:", hasattr(clf, "coef_"),
        "| feature count:",
        clf.n_features_in_ if hasattr(clf, "n_features_in_") else "Not fitted"
    )

model1 | fitted: True | feature count: 50000
threshold_model | fitted: True | feature count: 50000
glove_model | fitted: True | feature count: 100
sentence_model_lr | fitted: True | feature count: 384


In [80]:
import os
import joblib

SAVE_DIR = "/kaggle/working/amazon_sentiment_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the verified Bigram TF-IDF model pair
assert len(tfidf_bigram.get_feature_names_out()) == model1.n_features_in_

joblib.dump(
    tfidf_bigram,
    f"{SAVE_DIR}/bigram_tfidf_vectorizer.pkl"
)

joblib.dump(
    model1,
    f"{SAVE_DIR}/bigram_logistic_regression.pkl"
)

# Save the other trained classifiers
joblib.dump(
    glove_model,
    f"{SAVE_DIR}/glove_logistic_regression.pkl"
)

joblib.dump(
    sentence_model_lr,
    f"{SAVE_DIR}/sentence_transformer_logistic_regression.pkl"
)

print("Verified Bigram TF-IDF artefacts saved.")
print("Bigram classifier features:", model1.n_features_in_)
print(
    "Bigram vectorizer features:",
    len(tfidf_bigram.get_feature_names_out())
)


Verified Bigram TF-IDF artefacts saved.
Bigram classifier features: 50000
Bigram vectorizer features: 50000


In [81]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

SAVE_DIR = "/kaggle/working/amazon_sentiment_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# Build the summary dynamically from the predictions generated in this notebook.
results = pd.DataFrame([
    {
        "Model": "Majority Classifier",
        "Accuracy": accuracy_score(y_test, baseline_pred),
        "F1 Score": f1_score(y_test, baseline_pred, zero_division=0),
        "Evaluation": "Full test set"
    },
    {
        "Model": "Unigram TF-IDF + Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "Evaluation": "Full test set"
    },
    {
        "Model": "Bigram TF-IDF + Logistic Regression (default threshold)",
        "Accuracy": accuracy_score(y_test, y_pred_bigram),
        "F1 Score": f1_score(y_test, y_pred_bigram, zero_division=0),
        "Evaluation": "Full test set"
    },
    {
        "Model": f"Bigram TF-IDF + Logistic Regression (threshold = {best_threshold:.2f})",
        "Accuracy": accuracy_score(y_test, y_pred_best),
        "F1 Score": f1_score(y_test, y_pred_best, zero_division=0),
        "Evaluation": "Full test set; threshold selected on validation data"
    },
    {
        "Model": "GloVe + Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_glove),
        "F1 Score": f1_score(y_test, y_pred_glove, zero_division=0),
        "Evaluation": "Full test set"
    },
    {
        "Model": "Sentence Transformer + Logistic Regression",
        "Accuracy": accuracy_score(y_test_sample, y_test_pred_sentence),
        "F1 Score": f1_score(y_test_sample, y_test_pred_sentence, zero_division=0),
        "Evaluation": "2,000-review test subset"
    },
    {
        "Model": "Fine-tuned BERT",
        "Accuracy": accuracy_score(
            bert_results["Actual"],
            bert_results["Predicted"]
        ),
        "F1 Score": f1_score(
            bert_results["Actual"],
            bert_results["Predicted"],
            zero_division=0
        ),
        "Evaluation": "1,000-review validation subset"
    }
])

results.to_csv(f"{SAVE_DIR}/model_comparison.csv", index=False)

# Save BERT error analysis
bert_results.to_csv(
    f"{SAVE_DIR}/bert_error_analysis.csv",
    index=False
)

# Create artefact inventory
inventory = []

for root, dirs, files in os.walk(SAVE_DIR):
    for file in files:
        path = os.path.join(root, file)
        inventory.append({
            "File": os.path.relpath(path, SAVE_DIR),
            "Size_MB": round(os.path.getsize(path) / (1024**2), 2)
        })

pd.DataFrame(inventory).to_csv(
    f"{SAVE_DIR}/artefact_inventory.csv",
    index=False
)

print("PROJECT FILES SAVED")
print(results.to_string(index=False))
print("\nTotal saved files:", len(inventory))
print("\nComparison CSV and artefact inventory created.")


PROJECT FILES SAVED
                                                  Model  Accuracy  F1 Score                                           Evaluation
                                    Majority Classifier  0.780677  0.876832                                        Full test set
                   Unigram TF-IDF + Logistic Regression  0.901558  0.938246                                        Full test set
Bigram TF-IDF + Logistic Regression (default threshold)  0.928825  0.955136                                        Full test set
 Bigram TF-IDF + Logistic Regression (threshold = 0.52)  0.929247  0.955274 Full test set; threshold selected on validation data
                            GloVe + Logistic Regression  0.813987  0.888388                                        Full test set
             Sentence Transformer + Logistic Regression  0.833000  0.898788                             2,000-review test subset
                                        Fine-tuned BERT  0.903000  0.939413  

In [82]:
import shutil
import os

SAVE_DIR = "/kaggle/working/amazon_sentiment_models"
ZIP_PATH = "/kaggle/working/amazon_sentiment_models_backup"

shutil.make_archive(ZIP_PATH, "zip", SAVE_DIR)

final_zip = ZIP_PATH + ".zip"

print("ZIP CREATED SUCCESSFULLY")
print("File:", final_zip)
print("Size:", round(os.path.getsize(final_zip) / (1024**2), 2), "MB")

ZIP CREATED SUCCESSFULLY
File: /kaggle/working/amazon_sentiment_models_backup.zip
Size: 612.63 MB


In [83]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# Run the threshold-selection code cell and the following untouched-test
# evaluation code cell first. y_pred_best must contain predictions
# from the validation-selected threshold on the untouched test set.
results = []

def evaluate_model(name, actual, predicted, probabilities=None):
    row = {
        "Model": name,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1 Score": f1_score(actual, predicted, zero_division=0),
        "ROC-AUC": (
            roc_auc_score(actual, probabilities)
            if probabilities is not None else np.nan
        )
    }
    results.append(row)

evaluate_model("Unigram TF-IDF + Logistic Regression", y_test, y_pred)

evaluate_model(
    "Bigram TF-IDF + Logistic Regression (default threshold)",
    y_test, y_pred_bigram, positive_proba
)

# Threshold and predictions are generated by the preceding
# threshold-selection and untouched-test evaluation code cells.
evaluate_model(
    f"Bigram TF-IDF (validation-selected threshold = {best_threshold:.2f})",
    y_test, y_pred_best, test_positive_proba
)

evaluate_model("GloVe + Logistic Regression", y_test, y_pred_glove)

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values(
    "F1 Score", ascending=False
).reset_index(drop=True)

display(comparison_df.round(4))

comparison_df.to_csv(
    "/kaggle/working/model_comparison_full_test.csv",
    index=False
)

print("Comparison saved successfully.")


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Bigram TF-IDF (validation-selected threshold = 0.52),0.9292,0.9430,0.9679,0.9553,0.9675
1,Bigram TF-IDF + Logistic Regression (default threshold),0.9288,0.9402,0.9705,0.9551,0.9675
2,Unigram TF-IDF + Logistic Regression,0.9016,0.9194,0.9579,0.9382,NaN
3,GloVe + Logistic Regression,0.8140,0.8356,0.9483,0.8884,NaN


Comparison saved successfully.


In [84]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)
import pandas as pd

extra_results = []

# Sentence Transformer
extra_results.append({
    "Model": "Sentence Transformer + Logistic Regression",
    "Accuracy": accuracy_score(y_test_sample, y_test_pred_sentence),
    "Precision": precision_score(y_test_sample, y_test_pred_sentence),
    "Recall": recall_score(y_test_sample, y_test_pred_sentence),
    "F1 Score": f1_score(y_test_sample, y_test_pred_sentence),
    "ROC-AUC": None,
    "Evaluation Set": "2,000-review subset"
})

# Fine-tuned BERT
extra_results.append({
    "Model": "Fine-tuned BERT",
    "Accuracy": accuracy_score(bert_results["Actual"], bert_results["Predicted"]),
    "Precision": precision_score(bert_results["Actual"], bert_results["Predicted"]),
    "Recall": recall_score(bert_results["Actual"], bert_results["Predicted"]),
    "F1 Score": f1_score(bert_results["Actual"], bert_results["Predicted"]),
    "ROC-AUC": None,
    "Evaluation Set": "1,000-review validation subset"
})

extra_df = pd.DataFrame(extra_results)
display(extra_df.round(4))

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Evaluation Set
0,Sentence Transformer + Logistic Regression,0.833,0.8513,0.9519,0.8988,None,"2,000-review subset"
1,Fine-tuned BERT,0.903,0.9171,0.9629,0.9394,None,"1,000-review validation subset"


### Full-Test-Set Comparison: Results and Interpretation

The main comparison reports unigram TF-IDF, default-threshold bigram TF-IDF, GloVe, and the validation-selected-threshold bigram model on the held-out test set.

**Execution requirement:** run the threshold-selection code cell and the following untouched-test evaluation code cell before running the model-comparison code. The selected threshold and its test metrics must be generated in the current Kaggle session. Do not reuse previously stored optimized-threshold values.

The threshold is selected using only a validation split from the original training data. A temporary model is used for threshold selection, and the selected value is transferred to the final model trained on the complete original training set. The held-out test set is not used for threshold selection.

Sentence Transformer and fine-tuned BERT experiments use different evaluation subsets and are discussed separately; their metrics should not be ranked directly against the full-test-set results.

### Interpretation

- Compare the default-threshold bigram model with the validation-selected-threshold version using the newly generated metrics.
- Bigrams represent short phrases in addition to individual words.
- GloVe does not outperform TF-IDF in this implementation.
- Bigram TF-IDF with Logistic Regression remains the selected deployment model based on its reported performance and practical inference requirements.

Do not report a new threshold-adjusted F1-score until the threshold-selection and untouched-test evaluation code cells have been executed successfully in the Kaggle runtime.


In [85]:
import joblib
import os

SAVE_DIR = "/kaggle/working/amazon_sentiment_models"

# Verify matching feature dimensions
assert tfidf_bigram.transform(
    ["This product is excellent"]
).shape[1] == model1.n_features_in_

# Save the deployment components
joblib.dump(
    tfidf_bigram,
    f"{SAVE_DIR}/deployment_vectorizer.pkl"
)

joblib.dump(
    model1,
    f"{SAVE_DIR}/deployment_classifier.pkl"
)

print("Deployment model saved successfully!")

Deployment model saved successfully!


In [86]:
import joblib

SAVE_DIR = "/kaggle/working/amazon_sentiment_models"

vectorizer = joblib.load(
    f"{SAVE_DIR}/deployment_vectorizer.pkl"
)

classifier = joblib.load(
    f"{SAVE_DIR}/deployment_classifier.pkl"
)

def predict_sentiment(review):
    features = vectorizer.transform([review])
    prediction = classifier.predict(features)[0]
    probabilities = classifier.predict_proba(features)[0]

    return {
        "Sentiment": "Positive" if prediction == 1 else "Negative",
        "Negative Probability": round(float(probabilities[0]), 4),
        "Positive Probability": round(float(probabilities[1]), 4)
    }

test_reviews = [
    "This product is absolutely amazing. Great quality and excellent taste!",
    "Terrible product. It tastes awful and I will never buy it again."
]

for review in test_reviews:
    print("\nReview:", review)
    print("Prediction:", predict_sentiment(review))


Review: This product is absolutely amazing. Great quality and excellent taste!
Prediction: {'Sentiment': 'Positive', 'Negative Probability': 0.0004, 'Positive Probability': 0.9996}

Review: Terrible product. It tastes awful and I will never buy it again.
Prediction: {'Sentiment': 'Negative', 'Negative Probability': 0.9996, 'Positive Probability': 0.0004}


### Inference Verification: Interpretation

The saved artifacts are loaded and used to classify new review examples.

The expected output contains:

- The predicted sentiment label.
- The negative-class probability.
- The positive-class probability.

The probability values provide additional information about the model's prediction.

This verification is important because it tests the saved artifacts rather than only the original in-memory model.

A successful prediction confirms that the selected vectorizer and classifier can work together in an independent inference workflow.

In [87]:
import os
import joblib
import sklearn
import numpy
import scipy
import shutil

DEPLOY_DIR = "/kaggle/working/amazon_sentiment_deployment"
os.makedirs(DEPLOY_DIR, exist_ok=True)

# Copy only the two required model files
shutil.copy(
    "/kaggle/working/amazon_sentiment_models/deployment_vectorizer.pkl",
    f"{DEPLOY_DIR}/deployment_vectorizer.pkl"
)

shutil.copy(
    "/kaggle/working/amazon_sentiment_models/deployment_classifier.pkl",
    f"{DEPLOY_DIR}/deployment_classifier.pkl"
)

# Create Streamlit application
app_code = '''
import streamlit as st
import joblib
import os

st.set_page_config(
    page_title="Amazon Review Sentiment Analyzer",
    page_icon="🛒",
    layout="centered"
)

MODEL_DIR = os.path.dirname(os.path.abspath(__file__))

@st.cache_resource
def load_models():
    vectorizer = joblib.load(
        os.path.join(MODEL_DIR, "deployment_vectorizer.pkl")
    )
    classifier = joblib.load(
        os.path.join(MODEL_DIR, "deployment_classifier.pkl")
    )
    return vectorizer, classifier

vectorizer, classifier = load_models()

st.title("🛒 Amazon Review Sentiment Analyzer")

st.write(
    "Analyze customer reviews using a Bigram TF-IDF "
    "and Logistic Regression machine learning model."
)

st.markdown("---")

review = st.text_area(
    "Enter your product review:",
    placeholder="Example: This product is amazing. Great quality and taste!",
    height=150
)

if st.button("Analyze Sentiment", type="primary", use_container_width=True):

    if not review.strip():
        st.warning("Please enter a review first.")

    else:
        features = vectorizer.transform([review])
        prediction = classifier.predict(features)[0]
        probabilities = classifier.predict_proba(features)[0]

        negative_prob = float(probabilities[0])
        positive_prob = float(probabilities[1])

        st.markdown("### Prediction")

        if prediction == 1:
            st.success("😊 Positive Review")
        else:
            st.error("😞 Negative Review")

        st.markdown("### Prediction Confidence")

        col1, col2 = st.columns(2)

        with col1:
            st.metric(
                "Positive",
                f"{positive_prob * 100:.2f}%"
            )

        with col2:
            st.metric(
                "Negative",
                f"{negative_prob * 100:.2f}%"
            )

        st.markdown("### Probability Distribution")

        st.bar_chart({
            "Probability": {
                "Positive": positive_prob,
                "Negative": negative_prob
            }
        })

st.markdown("---")

st.caption(
    "Model: Bigram TF-IDF + Logistic Regression | "
    "Amazon Fine Food Reviews Dataset"
)
'''

with open(f"{DEPLOY_DIR}/app.py", "w") as f:
    f.write(app_code)

# Create compatible requirements
requirements = f"""streamlit>=1.35,<2
scikit-learn=={sklearn.__version__}
numpy=={numpy.__version__}
scipy=={scipy.__version__}
joblib=={joblib.__version__}
"""

with open(f"{DEPLOY_DIR}/requirements.txt", "w") as f:
    f.write(requirements)

# Package deployment files
shutil.make_archive(
    "/kaggle/working/amazon_sentiment_deployment",
    "zip",
    DEPLOY_DIR
)

print("DEPLOYMENT PACKAGE CREATED")
print("\nFiles:")
for file in os.listdir(DEPLOY_DIR):
    print(file)

print("\nLibrary versions:")
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Joblib:", joblib.__version__)

print("\nZIP: /kaggle/working/amazon_sentiment_deployment.zip")

DEPLOYMENT PACKAGE CREATED

Files:
deployment_classifier.pkl
app.py
deployment_vectorizer.pkl
requirements.txt

Library versions:
Scikit-learn: 1.6.1
NumPy: 2.1.3
SciPy: 1.16.3
Joblib: 1.6.0

ZIP: /kaggle/working/amazon_sentiment_deployment.zip


# 19. Conclusion and Future Scope

## Conclusion

This project developed and evaluated multiple approaches for binary sentiment classification using the Amazon Fine Food Reviews dataset.

The workflow began with dataset exploration, target construction, and a majority-class baseline before progressing through traditional machine learning and pretrained language representation techniques.

The experiments included:

- Unigram TF-IDF with Logistic Regression.
- Bigram TF-IDF with Logistic Regression.
- Classification threshold adjustment.
- GloVe word embeddings.
- Sentence Transformer embeddings.
- BERT contextual representations.
- Fine-tuning BERT for sentiment classification.

Among the models evaluated on the full held-out test set, Bigram TF-IDF with Logistic Regression achieved the strongest observed performance.

It achieved:

| Metric | Result |
|---|---:|
| Accuracy | 92.88% |
| Precision | 94.02% |
| Recall | 97.05% |
| F1-score | 95.51% |
| ROC-AUC | 96.75% |

The bigram representation improved on the unigram model, demonstrating the usefulness of phrase-level features in sentiment classification.

The project also investigated prediction confidence using fine-tuned BERT. A confidence-based manual-review mechanism was developed to identify uncertain predictions and capture a proportion of classification errors.

Finally, the selected bigram TF-IDF model and its matching Logistic Regression classifier were prepared for deployment through a Streamlit application.

## Limitations

1. The sentiment labels are derived from numerical ratings and may not perfectly represent the actual language expressed in every review.
2. Three-star reviews are grouped into the negative class, which may introduce label noise.
3. The dataset is imbalanced, with positive reviews representing approximately 78% of the records.
4. The Sentence Transformer and BERT experiments use smaller evaluation subsets than the full-test-set TF-IDF experiments.
5. The fine-tuned BERT experiment uses only one training epoch and a limited training subset.
6. The confidence threshold was selected from validation observations and should not be assumed to be universally optimal.

## Future Scope

1. Evaluate the models on a common, fixed test set for a more controlled comparison.
2. Investigate class-weighted training and other imbalance-handling strategies.
3. Experiment with more advanced text preprocessing and feature selection.
4. Fine-tune transformer models using larger training subsets and additional epochs.
5. Explore probability calibration and more systematic threshold selection.
6. Extend the application to analyse large collections of reviews and monitor sentiment trends.
7. Investigate explainability techniques to identify the words and phrases influencing model predictions.